# 03 · Baseline CNN trained from scratch

**ITRI626 · Colile Sibanda**

Objective **O3**. Runs on Colab; the laptop runs this notebook
only with `smoke.yaml`. The split is loaded from the committed files and
re-verified before a single batch is drawn.


In [ ]:
# On Colab only: mount Drive and install what the runtime is missing.
# from google.colab import drive; drive.mount('/content/drive')
# !pip -q install timm imagehash


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "baseline_cnn.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

## The baseline

Four convolutional blocks and a linear head, no pretrained weights. It is the bar the other two models have to clear, and it also measures the epoch time that sets the number of images per class.


In [ ]:
from docclf.splits.io import load_splits
from docclf.splits.leakage import assert_no_group_overlap
from docclf.data.loaders import build_dataloaders

splits = load_splits(paths.splits)
assert_no_group_overlap(splits, cfg.split.group_column)
loaders = build_dataloaders(splits, cfg)
{name: len(loader.dataset) for name, loader in loaders.items()}

In [ ]:
from docclf.models.registry import build_model, model_family
from docclf.models.summary import describe_model
from docclf.utils.env import select_device

model = build_model('baseline_cnn', cfg.n_classes, cfg)
device = select_device(cfg)
summary = describe_model(model, 'baseline_cnn')
print(f'Family: {model_family("baseline_cnn")}')
print(f'Device: {device}')
summary.as_row()

## Train

The loop is shared, unchanged, by all three models: whatever differs between them differs in the config, where it can be reported.


In [ ]:
from docclf.training.trainer import fit
from docclf.training.class_weights import compute_class_weights

weights = (
    compute_class_weights(splits[splits['split'] == 'train'], cfg.n_classes)
    if cfg.train.class_weights
    else None
)
history = fit(model, loaders, cfg, device, paths.checkpoints, weights)
history_path = history.save(paths.metrics)
history.to_frame()

## Learning curves


In [ ]:
from docclf.evaluation.curves import plot_learning_curves
from IPython.display import Image as Show

figure_path = plot_learning_curves(history.to_frame(), paths.figures / 'curves_baseline_cnn.png', 'baseline_cnn')
best = history.best(cfg.train.monitor)
print(f'Best epoch: {best.epoch}')
print(f'Best validation macro-F1: {best.val_macro_f1:.4f}')
print(f'Mean seconds per epoch: {sum(r.seconds for r in history.records) / len(history.records):.1f}')
Show(str(figure_path))

## Record the run

Add the row to `docs/experiment-log.md`: date, config, commit hash,
epochs, seconds per epoch, best validation macro-F1, and anything that
went wrong. A run that is not in the log did not happen.
